In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
train_df = pd.read_csv("trainingData.csv")

train_df.head()

test_df = pd.read_csv("validationData.csv")

test_df.head()

,WAP001,WAP002,WAP003,WAP004,WAP005,WAP006,WAP007,WAP008,WAP009,WAP010,...,WAP520,LONGITUDE,LATITUDE,FLOOR,BUILDINGID,SPACEID,RELATIVEPOSITION,USERID,PHONEID,TIMESTAMP
0,100,100,100,100,100,100,100,100,100,100,...,100,-7515.916799,4.864890e+06,1,1,0,0,0,0,1380872703
1,100,100,100,100,100,100,100,100,100,100,...,100,-7383.867221,4.864840e+06,4,2,0,0,0,13,1381155054
2,100,100,100,100,100,100,100,100,100,100,...,100,-7374.302080,4.864847e+06,4,2,0,0,0,13,1381155095
3,100,100,100,100,100,100,100,100,100,100,...,100,-7365.824883,4.864843e+06,4,2,0,0,0,13,1381155138
4,100,100,100,100,100,100,100,100,100,100,...,100,-7641.499303,4.864922e+06,2,0,0,0,0,2,1380877774


In [3]:
#Cleaning

# Training Data

# 1. Handling missing RSSI values

# Replace 100 with -110 (weak signal)
train_df.iloc[:, :520] = train_df.iloc[:, :520].replace(100, -110)

# 2. Separating  features & targets

X_train = train_df.iloc[:, :520]
y_class_train = train_df[['BUILDINGID', 'FLOOR']]
y_reg_train = train_df[['LONGITUDE', 'LATITUDE']]


In [5]:
#Cleaning

# TEST DATA

# 1. Handling missing RSSI values

test_df.iloc[:, :520] = test_df.iloc[:, :520].replace(100, -110)

# 2. Separating  features & targets

X_test = test_df.iloc[:, :520]
y_class_test = test_df[['BUILDINGID', 'FLOOR']]
y_reg_test = test_df[['LONGITUDE', 'LATITUDE']]




In [18]:
# Scaling 
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)  # ONLY transform, not fit

y_reg_train_scaled = scaler.fit_transform(y_reg_train)
y_reg_test_scaled = scaler.transform(y_reg_test)



In [7]:
#Traning models

In [20]:
#1) MLP Regressor (Regression)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

mlp = MLPRegressor(
    hidden_layer_sizes=(100,50),
    max_iter=500,
    early_stopping=True,
    learning_rate='adaptive',
    n_iter_no_change=20,
    random_state=42
)

# Train
mlp.fit(X_train, y_reg_train_scaled)

# Predict
y_pred_scaled = mlp.predict(X_test)
y_pred_mlp = scaler.inverse_transform(y_pred_scaled)

# Evaluate
rmse = np.sqrt(mean_squared_error(y_reg_test, y_pred_mlp))
mae = mean_absolute_error(y_reg_test, y_pred_mlp)

print("MLP RMSE:", rmse)
print("MLP MAE:", mae)

MLP RMSE: 17.121111323703957
MLP MAE: 9.862963130609891


In [ ]:
#will take a few mins to run the code
#code explanation

# For predicting the coordinates (longitude and latitude) from WiFi signals, we used an MLP Regressor. In this code, the target values were first scaled
# so that the neural network could learn more easily, because neural networks work better when both inputs and outputs are on similar scales. The model
# has two hidden layers with 100 and 50 neurons, and it uses an adaptive learning rate and early stopping to make sure it trains efficiently without 
# overfitting. After training on the scaled training data, predictions were made on the test data and then converted back to the original scale. 
# The performance was measured using RMSE and MAE to understand how close the predictions were to the actual coordinates. We did not use GridSearchCV here
# because tuning multiple hyperparameters for this model on a small dataset can take a lot of time and was not strictly required; the chosen parameters 
# are sufficient to achieve good results, and this approach also avoids complexity while keeping the code simple and understandable.

#evaluation

# MLP RMSE: 17.121111323703957
# MLP MAE: 9.862963130609891


In [15]:
 #2) KNN (Classification)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
import numpy as np

# =====================================================
# Custom scorer → fixes "multiclass-multioutput" warning
# =====================================================
def multioutput_accuracy(y_true, y_pred):
    return np.mean(np.all(y_true == y_pred, axis=1))

custom_scorer = make_scorer(multioutput_accuracy)

# =====================================================
# Grid Search for best K (optimization)
# =====================================================
param_grid_knn = {'n_neighbors': [3, 5, 7]}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=3,
    scoring=custom_scorer
)

grid_knn.fit(X_train, y_class_train)
best_knn = grid_knn.best_estimator_

print("Best K:", grid_knn.best_params_)

# =====================================================
# Predictions
# =====================================================
y_pred_knn = best_knn.predict(X_test)

building_pred = y_pred_knn[:, 0]
floor_pred = y_pred_knn[:, 1]

building_true = y_class_test['BUILDINGID'].values
floor_true = y_class_test['FLOOR'].values

# =====================================================
# BUILDING METRICS
# =====================================================
print("\n=== BUILDING METRICS ===")
print("Accuracy:", accuracy_score(building_true, building_pred))
print("Precision:", precision_score(building_true, building_pred, average='weighted'))
print("Recall:", recall_score(building_true, building_pred, average='weighted'))
print("F1 Score:", f1_score(building_true, building_pred, average='weighted'))

# =====================================================
# FLOOR METRICS
# =====================================================
print("\n=== FLOOR METRICS ===")
print("Accuracy:", accuracy_score(floor_true, floor_pred))
print("Precision:", precision_score(floor_true, floor_pred, average='weighted'))
print("Recall:", recall_score(floor_true, floor_pred, average='weighted'))
print("F1 Score:", f1_score(floor_true, floor_pred, average='weighted'))

# =====================================================
# COMBINED (Exact Location) ACCURACY
# =====================================================
combined_acc = np.mean(
    (building_pred == building_true) &
    (floor_pred == floor_true)
)

print("\n-----------------------------")
print("Combined Accuracy:", combined_acc)

Best K: {'n_neighbors': 7}

=== BUILDING METRICS ===
Accuracy: 0.990999099909991
Precision: 0.9910701229869544
Recall: 0.990999099909991
F1 Score: 0.9910118310859327

=== FLOOR METRICS ===
Accuracy: 0.8037803780378038
Precision: 0.8181954123902471
Recall: 0.8037803780378038
F1 Score: 0.8061863832620796

-----------------------------
Combined Accuracy: 0.8010801080108011


In [ ]:
# #code explanation
# This code trains a K-Nearest Neighbors (KNN) model to predict two things at the same time: building and floor from your data, then checks how well it performs.

# First, a custom scoring function is created because the model predicts two outputs together. Normal accuracy doesn’t work well for this, so the function only gives a correct score when both building AND floor are correct for the same sample. That fixes the multiclass-multioutput warning.

# Then GridSearchCV is used to automatically find the best value of K (number of neighbors) by testing 3, 5, and 7 using cross-validation. This helps pick the KNN model that performs best instead of guessing.

# After choosing the best model, it makes predictions on the test data. Since the model outputs two columns (building + floor), the code splits them so each can be evaluated separately.

# Next, it prints accuracy, precision, recall, and F1 score for:

# building prediction
# floor prediction

# Finally, it calculates a combined accuracy, which is the strictest metric — it only counts a prediction as correct if both building and floor are correct at the same time. This gives the real “exact location” performance of the model.


#EVALUATION
# Best K: {'n_neighbors': 7}

# === BUILDING METRICS ===
# Accuracy: 0.990999099909991
# Precision: 0.9910701229869544
# Recall: 0.990999099909991
# F1 Score: 0.9910118310859327

# === FLOOR METRICS ===
# Accuracy: 0.8037803780378038
# Precision: 0.8181954123902471
# Recall: 0.8037803780378038
# F1 Score: 0.8061863832620796

# -----------------------------
# Combined Accuracy: 0.8010801080108011

In [ ]:
12345